# Capítulo 8 — Taller integrador: Python aplicado a la ciencia de datos

**Serie Ciencia de Datos · Tomo 1 — Introducción al Data Science y Programación con Python**  
José Fernando Meza

Código del capítulo, extraído del libro. Repositorio: https://github.com/josefer76/serie-ciencia-de-datos

> Ejecute las celdas en orden: algunas dependen de variables definidas antes. Las celdas que usan `input()` esperan que usted escriba un valor.

## Código del capítulo

### 8.2 Cargar los datos de la tienda

*Texto del libro:*

```text
codigo,fecha,cliente,ciudad,categoria,producto,precio,cantidad,devuelto
P045,2026-02-26,carlos,Barranquilla,ropa,gorra,$ 35.000,1,si
P036,2026-04-28,sofia,Bogotá,ropa,chaqueta,$ 230.000,3,no
```

In [ ]:
import csv
import urllib.request

# 1. Descargar el archivo desde el repositorio del libro a la sesión de Colab.
URL_DATOS = ("https://raw.githubusercontent.com/josefer76/"
             "serie-ciencia-de-datos/main/libro1/datos/pedidos_tienda.csv")
urllib.request.urlretrieve(URL_DATOS, "pedidos_tienda.csv")

# 2. Abrir el archivo y leer cada fila como un diccionario.
with open("pedidos_tienda.csv", encoding="utf-8") as archivo:
    pedidos_crudos = list(csv.DictReader(archivo))

print(f"Filas cargadas: {len(pedidos_crudos)}")

# La primera fila, un dato por línea.
for columna, valor in pedidos_crudos[0].items():
    print(f"  {columna}: {valor!r}")

*Salida en el libro:*

```text
Filas cargadas: 64
  codigo: 'P045'
  fecha: '2026-02-26'
  cliente: 'carlos'
  ciudad: 'Barranquilla'
  categoria: 'ropa'
  producto: 'gorra'
  precio: '$ 35.000'
  cantidad: '1'
  devuelto: 'si'
```

## Soluciones de los ejercicios

### [Ex-L1-C8-01] — El informe de la tienda de Doña Marta

#### Parte B — Entender los datos (fase 2)

In [ ]:
import csv
import urllib.request

URL_DATOS = ("https://raw.githubusercontent.com/josefer76/"
             "serie-ciencia-de-datos/main/libro1/datos/pedidos_tienda.csv")
urllib.request.urlretrieve(URL_DATOS, "pedidos_tienda.csv")

with open("pedidos_tienda.csv", encoding="utf-8") as archivo:
    pedidos_crudos = list(csv.DictReader(archivo))

# --- Primer vistazo ---
print(f"Filas: {len(pedidos_crudos)}")
columnas = list(pedidos_crudos[0])
print(f"Columnas ({len(columnas)}):")
print("  " + ", ".join(columnas[:5]))
print("  " + ", ".join(columnas[5:]))

# --- Ciudades tal como llegaron: un conjunto da los valores distintos ---
ciudades_crudas = {pedido["ciudad"] for pedido in pedidos_crudos}
print(f"Formas distintas de escribir las ciudades: {len(ciudades_crudas)}")

# --- Códigos repetidos: conjunto de vistos, un solo recorrido ---
codigos_vistos = set()
codigos_repetidos = set()
for pedido in pedidos_crudos:
    if pedido["codigo"] in codigos_vistos:
        codigos_repetidos.add(pedido["codigo"])
    codigos_vistos.add(pedido["codigo"])
print(f"Códigos repetidos: {sorted(codigos_repetidos)}")

# --- Precios que no son un número positivo ---
precios_problema = []
for pedido in pedidos_crudos:
    texto = pedido["precio"].replace("$", "").replace(".", "").strip()
    try:
        if int(texto) <= 0:
            precios_problema.append(pedido["precio"])
    except ValueError:
        precios_problema.append(pedido["precio"])
print(f"Precios con problemas: {precios_problema}")

# --- Cantidades raras ---
cantidades_raras = [pedido["cantidad"] for pedido in pedidos_crudos
                    if not 1 <= int(pedido["cantidad"]) <= 100]
print(f"Cantidades fuera de 1 a 100: {cantidades_raras}")

*Salida en el libro:*

```text
Filas: 64
Columnas (9):
  codigo, fecha, cliente, ciudad, categoria
  producto, precio, cantidad, devuelto
Formas distintas de escribir las ciudades: 11
Códigos repetidos: ['P004', 'P016', 'P028', 'P045']
Precios con problemas: ['consultar', '-45000', '', '-89000', '']
Cantidades fuera de 1 a 100: ['250', '0']
```

#### Parte C — Preparar los datos (fase 3)

In [ ]:
def limpiar_ciudad(ciudad):
    """Devuelve la ciudad sin espacios de más, en minúsculas y sin tildes."""
    ciudad = ciudad.strip().lower()
    for con_tilde, sin_tilde in zip("áéíóú", "aeiou"):
        ciudad = ciudad.replace(con_tilde, sin_tilde)
    return ciudad


def limpiar_precio(texto):
    """Convierte un precio en formato colombiano ('$ 1.250.000') en entero."""
    return int(texto.replace("$", "").replace(".", "").strip())


def preparar_pedidos(pedidos_crudos):
    """Limpia los pedidos y descarta los que no sirven para el análisis.

    Devuelve una tupla (pedidos_limpios, descartes), donde descartes es un
    diccionario que cuenta cuántos pedidos se descartaron por cada motivo.
    """
    pedidos_limpios = []
    descartes = {}
    codigos_vistos = set()

    for pedido in pedidos_crudos:
        # 1. Repetidos: se conserva solo la primera aparición de cada código.
        if pedido["codigo"] in codigos_vistos:
            descartes["repetido"] = descartes.get("repetido", 0) + 1
            continue
        codigos_vistos.add(pedido["codigo"])

        # 2. Precio: debe poder convertirse y ser positivo.
        try:
            precio = limpiar_precio(pedido["precio"])
        except ValueError:
            descartes["precio no numérico"] = (
                descartes.get("precio no numérico", 0) + 1)
            continue
        if precio <= 0:
            descartes["precio negativo"] = (
                descartes.get("precio negativo", 0) + 1)
            continue

        # 3. Cantidad: entre 1 y 100.
        cantidad = int(pedido["cantidad"])
        if not 1 <= cantidad <= 100:
            descartes["cantidad fuera de rango"] = (
                descartes.get("cantidad fuera de rango", 0) + 1)
            continue

        # 4. Pedido limpio, con los tipos correctos y columnas nuevas.
        pedidos_limpios.append({
            "codigo": pedido["codigo"],
            "mes": pedido["fecha"][:7],          # "2026-03-15" -> "2026-03"
            "cliente": pedido["cliente"],
            "ciudad": limpiar_ciudad(pedido["ciudad"]),
            "categoria": pedido["categoria"],
            "producto": pedido["producto"],
            "precio": precio,
            "cantidad": cantidad,
            "valor": precio * cantidad,
            "devuelto": pedido["devuelto"] == "si",
        })

    return pedidos_limpios, descartes


pedidos, descartes = preparar_pedidos(pedidos_crudos)
print(f"Pedidos limpios: {len(pedidos)}")
print("Descartes por motivo:")
for motivo, cantidad in descartes.items():
    print(f"  {motivo}: {cantidad}")
print("Ciudades después de limpiar:")
print(sorted({pedido["ciudad"] for pedido in pedidos}))

*Salida en el libro:*

```text
Pedidos limpios: 53
Descartes por motivo:
  repetido: 4
  precio no numérico: 3
  precio negativo: 2
  cantidad fuera de rango: 2
Ciudades después de limpiar:
['barranquilla', 'bogota', 'cali', 'cartagena', 'medellin']
```

#### Parte D — Analizar (fase 4)

In [ ]:
def sumar_por(pedidos, columna):
    """Suma el valor de los pedidos no devueltos, agrupado por una columna."""
    totales = {}
    for pedido in pedidos:
        if not pedido["devuelto"]:
            clave = pedido[columna]
            totales[clave] = totales.get(clave, 0) + pedido["valor"]
    return totales


def ordenar_de_mayor_a_menor(diccionario):
    """Devuelve los pares (clave, valor) ordenados de mayor a menor valor."""
    return sorted(diccionario.items(), key=lambda par: par[1], reverse=True)


# 1. Ventas por ciudad y por categoría
ventas_por_ciudad = sumar_por(pedidos, "ciudad")
ventas_por_categoria = sumar_por(pedidos, "categoria")

# 2. Los cinco productos que más dinero trajeron
top_productos = ordenar_de_mayor_a_menor(sumar_por(pedidos, "producto"))[:5]

# 3. Porcentaje de devoluciones por categoría (sobre todos los pedidos)
pedidos_por_categoria = {}
devueltos_por_categoria = {}
for pedido in pedidos:
    categoria = pedido["categoria"]
    pedidos_por_categoria[categoria] = (
        pedidos_por_categoria.get(categoria, 0) + 1)
    if pedido["devuelto"]:
        devueltos_por_categoria[categoria] = (
            devueltos_por_categoria.get(categoria, 0) + 1)

porcentaje_devoluciones = {
    categoria: devueltos_por_categoria.get(categoria, 0) / total * 100
    for categoria, total in pedidos_por_categoria.items()
}

# 4. Clientes con cinco pedidos o más
pedidos_por_cliente = {}
for pedido in pedidos:
    cliente = pedido["cliente"]
    pedidos_por_cliente[cliente] = pedidos_por_cliente.get(cliente, 0) + 1
clientes_frecuentes = sorted(cliente for cliente, numero
                             in pedidos_por_cliente.items() if numero >= 5)

print("1. Ventas por ciudad:")
for ciudad, valor in ordenar_de_mayor_a_menor(ventas_por_ciudad):
    print(f"   {ciudad}: {valor}")
print("1. Ventas por categoría:")
for categoria, valor in ordenar_de_mayor_a_menor(ventas_por_categoria):
    print(f"   {categoria}: {valor}")
print("2. Los cinco productos que más dinero trajeron:")
for producto, valor in top_productos:
    print(f"   {producto}: {valor}")
print("3. Porcentaje de pedidos devueltos:")
for categoria, porcentaje in porcentaje_devoluciones.items():
    print(f"   {categoria}: {porcentaje:.1f} %")
print("4. Clientes con cinco pedidos o más:")
print(f"   {', '.join(clientes_frecuentes)}")

*Salida en el libro:*

```text
1. Ventas por ciudad:
   cartagena: 3100000
   bogota: 2622000
   cali: 2597000
   barranquilla: 1687000
   medellin: 435000
1. Ventas por categoría:
   electronica: 4380000
   ropa: 3765000
   hogar: 2296000
2. Los cinco productos que más dinero trajeron:
   parlante: 2730000
   chaqueta: 2300000
   audifonos: 1200000
   sarten: 979000
   jean: 840000
3. Porcentaje de pedidos devueltos:
   ropa: 38.1 %
   electronica: 7.7 %
   hogar: 5.3 %
4. Clientes con cinco pedidos o más:
   andres, carlos, lucia, luis, paula, sofia
```

#### Parte E — Evaluar (fase 5)

In [ ]:
# Comprobación 1: ninguna fila se perdió ni se contó dos veces.
total_descartados = sum(descartes.values())
print(f"Limpios + descartados = {len(pedidos) + total_descartados} "
      f"(filas originales: {len(pedidos_crudos)})")

# Comprobación 2: el total por ciudad y el total por categoría coinciden.
total_ventas = sum(pedido["valor"] for pedido in pedidos
                   if not pedido["devuelto"])
print(sum(ventas_por_ciudad.values()) == total_ventas)
print(sum(ventas_por_categoria.values()) == total_ventas)

*Salida en el libro:*

```text
Limpios + descartados = 64 (filas originales: 64)
True
True
```

#### Parte F — Poner en uso (fase 6)

In [ ]:
def mostrar_informe(pedidos, descartes):
    """Muestra el informe de ventas para Doña Marta."""
    print("INFORME DE VENTAS — ENERO A JUNIO DE 2026")
    print(f"Pedidos analizados: {len(pedidos)} "
          f"(descartados: {sum(descartes.values())})")
    total = sum(pedido["valor"] for pedido in pedidos
                if not pedido["devuelto"])
    print(f"Ventas sin devoluciones: {total} pesos")

    print("\nVentas por ciudad:")
    for ciudad, valor in ordenar_de_mayor_a_menor(
            sumar_por(pedidos, "ciudad")):
        print(f"  {ciudad.title():<13}{valor:>10} pesos")

    print("\nLos cinco productos que más dinero trajeron:")
    productos = ordenar_de_mayor_a_menor(sumar_por(pedidos, "producto"))
    for posicion, (producto, valor) in enumerate(productos[:5], start=1):
        print(f"  {posicion}. {producto:<11}{valor:>10} pesos")

    print("\nPedidos devueltos por categoría:")
    for categoria, porcentaje in porcentaje_devoluciones.items():
        print(f"  {categoria:<13}{porcentaje:>5.1f} %")

    print(f"\nClientes con cinco pedidos o más: "
          f"{', '.join(clientes_frecuentes)}")


mostrar_informe(pedidos, descartes)

*Salida en el libro:*

```text
INFORME DE VENTAS — ENERO A JUNIO DE 2026
Pedidos analizados: 53 (descartados: 11)
Ventas sin devoluciones: 10441000 pesos

Ventas por ciudad:
  Cartagena       3100000 pesos
  Bogota          2622000 pesos
  Cali            2597000 pesos
  Barranquilla    1687000 pesos
  Medellin         435000 pesos

Los cinco productos que más dinero trajeron:
  1. parlante      2730000 pesos
  2. chaqueta      2300000 pesos
  3. audifonos     1200000 pesos
  4. sarten         979000 pesos
  5. jean           840000 pesos

Pedidos devueltos por categoría:
  ropa          38.1 %
  electronica    7.7 %
  hogar          5.3 %

Clientes con cinco pedidos o más: andres, carlos, lucia, luis, paula, sofia
```

### [Ex-L1-C8-02] — Proyecto de ampliación I: recomendador de productos

In [ ]:
def indice_jaccard(conjunto_a, conjunto_b):
    """Devuelve el índice de Jaccard entre dos conjuntos (de 0 a 1)."""
    union = conjunto_a | conjunto_b
    if len(union) == 0:
        return 0.0
    return len(conjunto_a & conjunto_b) / len(union)


# --- 1. Productos comprados por cada cliente (sin contar los devueltos) ---
productos_por_cliente = {}
for pedido in pedidos:
    if not pedido["devuelto"]:
        cliente = pedido["cliente"]
        productos_por_cliente.setdefault(cliente, set()).add(
            pedido["producto"])


# --- 2. Cliente más parecido y recomendación ---
def recomendar(cliente, productos_por_cliente):
    """Devuelve (cliente más parecido, índice, productos a recomendar)."""
    if cliente not in productos_por_cliente:
        raise ValueError(f"El cliente '{cliente}' no tiene compras.")
    propios = productos_por_cliente[cliente]
    otros = [otro for otro in productos_por_cliente if otro != cliente]
    mas_parecido = max(
        otros,
        key=lambda otro: indice_jaccard(propios, productos_por_cliente[otro]))
    indice = indice_jaccard(propios, productos_por_cliente[mas_parecido])
    sugerencias = sorted(productos_por_cliente[mas_parecido] - propios)
    return mas_parecido, indice, sugerencias


for cliente in ["ana", "luis", "marta"]:
    parecido, indice, sugerencias = recomendar(cliente,
                                               productos_por_cliente)
    print(f"{cliente}: más parecido a {parecido} ({indice:.2f}); "
          f"recomendar {sugerencias}")

# --- 3. Cliente sin compras ---
try:
    recomendar("cliente_nuevo", productos_por_cliente)
except ValueError as error:
    print(f"Error: {error}")

*Salida en el libro:*

```text
ana: más parecido a felipe (0.50); recomendar ['chaqueta', 'cojin']
luis: más parecido a andres (0.38); recomendar ['audifonos', 'olla']
marta: más parecido a carlos (0.50); recomendar ['toalla']
Error: El cliente 'cliente_nuevo' no tiene compras.
```

### [Ex-L1-C8-03] — Proyecto de ampliación II: las ventas mes a mes

In [ ]:
# --- 1. Ventas por mes (sin devoluciones) ---
ventas_por_mes = sumar_por(pedidos, "mes")

# --- 2. Tabla y gráfico de barras en texto, en orden de fecha ---
PESOS_POR_ASTERISCO = 100000
mes_anterior = None
for mes in sorted(ventas_por_mes):
    valor = ventas_por_mes[mes]
    barra = "*" * (valor // PESOS_POR_ASTERISCO)
    if mes_anterior is None:
        cambio = "   (primer mes)"
    else:
        variacion = ((valor - ventas_por_mes[mes_anterior])
                     / ventas_por_mes[mes_anterior] * 100)
        cambio = f"{variacion:+7.1f} %"
    print(f"{mes}  {valor:>8} {cambio}  {barra}")
    mes_anterior = mes

# --- 3. Mejor y peor mes ---
print(f"Mejor mes: {max(ventas_por_mes, key=ventas_por_mes.get)}")
print(f"Peor mes: {min(ventas_por_mes, key=ventas_por_mes.get)}")

*Salida en el libro:*

```text
2026-01   1952000    (primer mes)  *******************
2026-02   2276000   +16.6 %  **********************
2026-03   1238000   -45.6 %  ************
2026-04   2850000  +130.2 %  ****************************
2026-05    717000   -74.8 %  *******
2026-06   1408000   +96.4 %  **************
Mejor mes: 2026-04
Peor mes: 2026-05
```